In [1]:
#pip install torch torchvision torchaudio

In [2]:
import torch
from torch import nn


In [ ]:
# setup for my m1pro laptop
if torch.backends.mps.is_available():
    device = torch.device("mps") # Apple Silicon GPU
elif torch.cuda.is_available():
    device = torch.device("cuda") # NVIDIA GPU - if available
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [ ]:
import torch
from torch import nn

class BrainTumorCNN(nn.Module):
    """
    Improved CNN for brain tumor classification.

    Key differences vs old SimpleCNN:
    - Multi-scale first block: 3x3 and 5x5 conv branches in parallel.
    - Deeper conv blocks so receptive field covers larger brain regions (better for asymmetry).
    - Texture-focused final block with more channels.
    - Global Average Pooling instead of huge Flatten+Linear (fewer params).
    - Still lightweight enough to train on an M1 Pro chip.
    """
    def __init__(self, in_channels: int = 3, num_classes: int = 1):
        super().__init__()

        # -------- Block 1: Multi-scale feature extraction --------
        # 3x3 conv (fine edges, boundaries)
        self.branch3x3 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        # Branch B: 5x5 conv (larger structures, coarse tumor shape)
        self.branch5x5 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=5, padding=2),
            nn.ReLU()
        )

        # After concatenation, channels = 16 + 16 = 32
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 64x64 -> 32x32

        # -------- Block 2: Deeper receptive field --------
        # Two stacked 3x3 convs increase effective receptive field
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # 32x32 -> 16x16
        )

        # -------- Block 3: Texture-focused block --------
        # More channels, still 3x3, no more spatial downsampling
        # Designed to learn tumor texture (heterogeneity, contrast)
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # -------- Global Average Pooling + Classifier --------
        # Global Average Pooling: each channel -> 1 scalar (how strong that pattern is anywhere)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))  # (N, C, H, W) -> (N, C, 1, 1)

        self.classifier = nn.Sequential(
            nn.Flatten(),               # (N, 128, 1, 1) -> (N, 128)
            nn.Dropout(p=0.5),          # Regularization (important for small-ish dataset)
            nn.Linear(128, 1)           # Single logit for BCEWithLogitsLoss
        )

    def forward(self, x):
        # x: (N, 3, 64, 64)

        # Multi-scale block
        x3 = self.branch3x3(x)        
        x5 = self.branch5x5(x)        
        x = torch.cat([x3, x5], dim=1)

        x = self.pool1(x)             

        # Deeper block
        x = self.block2(x)            

        # Texture block
        x = self.block3(x)            

        # Global average pooling
        x = self.global_pool(x)       

        # Classifier -> logit
        x = self.classifier(x)         # (N, 1)

        return x

In [5]:
model = BrainTumorCNN(in_channels=3, num_classes=1).to(device)
print(model)

BrainTumorCNN(
  (branch3x3): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
  )
  (branch5x5): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (global_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, i

In [15]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [16]:
data_path = "../dataset_with_label"

In [ ]:
# -------- Transforms --------
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# ---- Load ImageFolder datasets ----
train_data = datasets.ImageFolder(root=f"{data_path}/train", transform=train_transform)
test_data  = datasets.ImageFolder(root=f"{data_path}/test",  transform=test_transform)

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

train_dataloader = DataLoader(train_data, batch_size=16, shuffle=True)
test_dataloader  = DataLoader(test_data,  batch_size=16, shuffle=False)

print("Dataloaders ready.")

Train samples: 3009
Test samples: 753
Dataloaders ready.


In [ ]:
class BrainTumorCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=1):
        super().__init__()

        # multiple filter sizes in the first block(kernal size 3 and 5)
        # those 2 kernal will slide over the image in parallel to extract features at different scales
        self.branch3x3 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.branch5x5 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=5, padding=2),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2, 2)

        # second block(kernal) with increased detection field
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # a very large number of channels to focus on texture which is the overall of the image
        # eg. find smoothness, roughness, regular patterns
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # combined with global average pooling and dropout to reduce overfitting
        # so each channel is summarized into one value indicating the presence of that feature anywhere in the image
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x3 = self.branch3x3(x)
        x5 = self.branch5x5(x)
        x = torch.cat([x3, x5], dim=1)

        x = self.pool1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

In [19]:
# ----- Single epoch training -----
def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss, total_acc = 0, 0
    start = time.time()

    for X, y in dataloader:
        X, y = X.to(device), y.to(device).float()

        logits = model(X).squeeze(1)
        loss = loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (torch.sigmoid(logits) > 0.5).int()
        total_acc += (preds == y.int()).sum().item() / len(y)

    return total_loss / len(dataloader), total_acc / len(dataloader), time.time() - start


# ----- Single epoch testing -----
def test_step(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, total_acc = 0, 0
    start = time.time()

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device).float()
            logits = model(X).squeeze(1)
            loss = loss_fn(logits, y)

            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).int()
            total_acc += (preds == y.int()).sum().item() / len(y)

    return total_loss / len(dataloader), total_acc / len(dataloader), time.time() - start


# ----- Train Loop -----
def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, device, epochs=10):
    for epoch in range(epochs):
        train_loss, train_acc, t_time = train_step(model, train_dataloader, loss_fn, optimizer, device)
        test_loss, test_acc, v_time  = test_step(model, test_dataloader, loss_fn, device)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train: loss={train_loss:.4f}, acc={train_acc:.4f}, time={t_time:.2f}s | "
            f"Test: loss={test_loss:.4f}, acc={test_acc:.4f}, time={v_time:.2f}s"
        )

In [20]:
model = BrainTumorCNN().to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=10
)

Epoch 1/10 | Train: loss=0.5876, acc=0.6852, time=11.14s | Test: loss=0.4107, acc=0.8372, time=0.89s
Epoch 2/10 | Train: loss=0.4743, acc=0.7999, time=5.75s | Test: loss=0.3980, acc=0.8177, time=0.73s
Epoch 3/10 | Train: loss=0.3642, acc=0.8502, time=5.39s | Test: loss=0.3392, acc=0.8542, time=0.73s
Epoch 4/10 | Train: loss=0.2695, acc=0.8892, time=5.16s | Test: loss=0.3004, acc=0.8945, time=0.73s
Epoch 5/10 | Train: loss=0.2259, acc=0.9236, time=5.06s | Test: loss=0.1980, acc=0.9206, time=0.77s
Epoch 6/10 | Train: loss=0.2070, acc=0.9306, time=5.07s | Test: loss=0.1985, acc=0.9349, time=0.73s
Epoch 7/10 | Train: loss=0.1884, acc=0.9398, time=5.10s | Test: loss=0.1573, acc=0.9414, time=0.76s
Epoch 8/10 | Train: loss=0.1780, acc=0.9421, time=5.26s | Test: loss=0.1380, acc=0.9570, time=0.78s
Epoch 9/10 | Train: loss=0.1503, acc=0.9537, time=5.21s | Test: loss=0.1733, acc=0.9388, time=0.76s
Epoch 10/10 | Train: loss=0.1571, acc=0.9474, time=5.34s | Test: loss=0.1417, acc=0.9440, time=0.79